<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [ ]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [ ]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Создаем датасет для предобработки данных

In [ ]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Стандартные параметры нормализации ImageNet
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

# Трансформации для обучающей выборки
train_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        # HorizontalFlip автоматически меняет и координаты bbox'ов!
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'])
)

# Трансформации для тестовой/валидационной выборки
test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ]
)


Не забываем инициализировать наш датасет

In [ ]:
df_train.head()

,image_id,image,width,height,objects
0,311,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,640,640,"{'id': [573, 574], 'area': [1748, 15756], 'bbo..."
1,67,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,640,640,"{'id': [137, 138], 'area': [14136, 88392], 'bb..."
2,161,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,640,640,"{'id': [301, 302], 'area': [3016, 33768], 'bbo..."
3,210,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,640,640,"{'id': [379, 380], 'area': [832, 9248], 'bbox'..."
4,142,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,640,640,"{'id': [272, 273], 'area': [1520, 16704], 'bbo..."


In [ ]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [ ]:
import timm
import torch.nn as nn


class Backbone(nn.Module):

    def __init__(
        self,
        model_name="efficientnet_b0",
        out_indices=(-1, -2, -3),
        unfreeze_k_blocks=2,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True,
            out_indices=out_indices,
        )

        for param in self.backbone.parameters():
            param.requires_grad = False

        if unfreeze_k_blocks > 0:
            if hasattr(self.backbone, "blocks"):
                blocks = self.backbone.blocks
                for block in blocks[-unfreeze_k_blocks:]:
                    for param in block.parameters():
                        param.requires_grad = True

    def forward(self, x):
        return self.backbone(x)


### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Neck(nn.Module):
    def __init__(self, in_channels_list, out_channels):
        super().__init__()
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_ch, out_channels, kernel_size=1)
            for in_ch in in_channels_list
        ])

        self.fpn_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU()
            )
            for _ in in_channels_list
        ])

    def forward(self, features):
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]
        for i in range(len(laterals) - 2, -1, -1):
            upsampled = F.interpolate(
                laterals[i + 1],
                size=laterals[i].shape[-2:],
                mode='nearest'
            )
            laterals[i] = laterals[i] + upsampled

        outputs = [conv(lat) for conv, lat in zip(self.fpn_convs, laterals)]

        return outputs

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [ ]:
class ConfidenceFreeHeadZero(nn.Module):
    def __init__(self, in_channels, num_anchors, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.num_anchors = num_anchors

        self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.cls_head = nn.Conv2d(in_channels, num_anchors * num_classes, kernel_size=1)
        self.reg_head = nn.Conv2d(in_channels, num_anchors * 4, kernel_size=1)  # ТОЛЬКО 4!

    def forward(self, x):
        x = F.relu(self.conv(x))
        cls_logits = self.cls_head(x)
        bbox_preds = self.reg_head(x)
        return cls_logits, bbox_preds

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [ ]:
class Detector(nn.Module):
    def __init__(
        self,
        backbone_model_name="efficientnet_b0",
        neck_n_channels=256,
        num_classes=4,
        anchor_sizes=(32, 64, 128),
        anchor_ratios=(0.5, 1.0, 2.0),
        input_size=(640, 640),
    ):
        super().__init__()
        self.num_classes = num_classes

        self.backbone = Backbone(backbone_model_name, out_indices=(-1,))
        in_channels = self.backbone.backbone.feature_info.channels()[0]

        self.neck = Neck([in_channels], out_channels=neck_n_channels)

        num_anchors = len(anchor_sizes) * len(anchor_ratios)
        self.head = ConfidenceFreeHeadZero(
            in_channels=neck_n_channels,
            num_anchors=num_anchors,
            num_classes=num_classes
        )

        anchor_generator = AnchorGenerator(
            sizes=(anchor_sizes,),
            aspect_ratios=(anchor_ratios,)
        )
        reduction = self.backbone.backbone.feature_info.reduction()[0]
        grid_sizes = [[input_size[0] // reduction, input_size[1] // reduction]]

        anchors = anchor_generator.grid_anchors(
            grid_sizes,
            strides=[[reduction, reduction]]
        )
        anchors = torch.stack(anchors, dim=0)

        anchor_centers = (anchors[:, :, :2] + anchors[:, :, 2:]) / 2
        anchor_sizes_wh = anchors[:, :, 2:] - anchors[:, :, :2]

        self.register_buffer("anchors", anchors)
        self.register_buffer("anchor_centers", anchor_centers)
        self.register_buffer("anchor_sizes", anchor_sizes_wh)

    def forward(self, x):
        features = self.backbone(x)
        neck_features = self.neck(features)[0]  # [0] — один тензор, не список

        cls_logits, bbox_preds = self.head(neck_features)

        N = x.shape[0]

        cls_logits = cls_logits.permute(0, 2, 3, 1).contiguous()
        cls_logits = cls_logits.view(N, -1, self.num_classes)

        bbox_preds = bbox_preds.permute(0, 2, 3, 1).contiguous()
        bbox_offsets = bbox_preds.view(N, -1, 4)

        if self.training:
            return bbox_offsets, cls_logits

        bboxes = self.decode_bboxes(bbox_offsets)
        cls_probs = torch.sigmoid(cls_logits)

        return bboxes, cls_probs

    def decode_bboxes(self, bbox_offsets):
        tx = bbox_offsets[:, :, 0]
        ty = bbox_offsets[:, :, 1]
        tw = bbox_offsets[:, :, 2]
        th = bbox_offsets[:, :, 3]

        center_x = self.anchor_centers[:, :, 0] + torch.sigmoid(tx) * self.anchor_sizes[:, :, 0]
        center_y = self.anchor_centers[:, :, 1] + torch.sigmoid(ty) * self.anchor_sizes[:, :, 1]

        w = torch.exp(tw.clamp(max=10)) * self.anchor_sizes[:, :, 0]
        h = torch.exp(th.clamp(max=10)) * self.anchor_sizes[:, :, 1]

        x1 = center_x - w / 2
        y1 = center_y - h / 2
        x2 = center_x + w / 2
        y2 = center_y + h / 2

        return torch.stack([x1, y1, x2, y2], dim=-1)

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [ ]:
def TAL_assigner(
    pred_bboxes,
    pred_scores,
    gt_bboxes,
    gt_labels,
    anchor_centers,
    alpha=6.0,
    beta=1.0,
    topk=13,
    device='cuda'
):
    num_gt = gt_bboxes.size(0)
    num_bboxes = pred_bboxes.size(0)

    assigned_gt_inds = torch.zeros(num_bboxes, dtype=torch.long, device=device)
    assigned_labels = torch.full((num_bboxes,), -1, dtype=torch.long, device=device)
    max_overlaps = torch.zeros(num_bboxes, device=device)

    if num_gt == 0 or num_bboxes == 0:
        return assigned_gt_inds, assigned_labels, max_overlaps

    overlaps = bbox_iou(pred_bboxes, gt_bboxes)
    gt_scores = pred_scores[:, gt_labels]
    alignment_metrics = (gt_scores ** alpha) * (overlaps ** beta)

    cx = anchor_centers[:, 0].unsqueeze(1)
    cy = anchor_centers[:, 1].unsqueeze(1)

    gt_x1 = gt_bboxes[:, 0].unsqueeze(0)
    gt_y1 = gt_bboxes[:, 1].unsqueeze(0)
    gt_x2 = gt_bboxes[:, 2].unsqueeze(0)
    gt_y2 = gt_bboxes[:, 3].unsqueeze(0)

    inside_gt = (cx > gt_x1) & (cy > gt_y1) & (cx < gt_x2) & (cy < gt_y2)
    alignment_metrics = alignment_metrics * inside_gt.float()

    topk_actual = min(topk, num_bboxes)
    _, candidate_idx = alignment_metrics.topk(topk_actual, dim=0, largest=True)

    candidate_mask = torch.zeros(num_bboxes, num_gt, dtype=torch.bool, device=device)
    for gt_idx in range(num_gt):
        candidate_mask[candidate_idx[:, gt_idx], gt_idx] = True

    for pred_idx in range(num_bboxes):
        gt_indices = torch.where(candidate_mask[pred_idx])[0]
        if gt_indices.numel() > 1:
            best_gt = overlaps[pred_idx, gt_indices].argmax()
            best_gt_idx = gt_indices[best_gt]
            for gt_idx in gt_indices:
                if gt_idx != best_gt_idx:
                    candidate_mask[pred_idx, gt_idx] = False

    for gt_idx in range(num_gt):
        pos_preds = torch.where(candidate_mask[:, gt_idx])[0]
        if pos_preds.numel() > 0:
            assigned_gt_inds[pos_preds] = gt_idx + 1
            assigned_labels[pos_preds] = gt_labels[gt_idx]
            max_overlaps[pos_preds] = overlaps[pos_preds, gt_idx]

    return assigned_gt_inds, assigned_labels, max_overlaps


def bbox_iou(bboxes1, bboxes2, eps=1e-6):
    area1 = (bboxes1[:, 2] - bboxes1[:, 0]) * (bboxes1[:, 3] - bboxes1[:, 1])
    area2 = (bboxes2[:, 2] - bboxes2[:, 0]) * (bboxes2[:, 3] - bboxes2[:, 1])

    lt = torch.max(bboxes1[:, None, :2], bboxes2[None, :, :2])
    rb = torch.min(bboxes1[:, None, 2:], bboxes2[None, :, 2:])

    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]

    union = area1[:, None] + area2[None, :] - inter + eps

    iou = inter / union
    return iou

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [ ]:
from torchvision.ops import distance_box_iou_loss

In [ ]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [ ]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [ ]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 0.9940983653068542


In [ ]:
def diou_loss(pred_boxes, gt_boxes, eps=1e-7):
    px1, py1, px2, py2 = pred_boxes[:, 0], pred_boxes[:, 1], pred_boxes[:, 2], pred_boxes[:, 3]
    gx1, gy1, gx2, gy2 = gt_boxes[:, 0], gt_boxes[:, 1], gt_boxes[:, 2], gt_boxes[:, 3]

    p_area = (px2 - px1) * (py2 - py1)
    g_area = (gx2 - gx1) * (gy2 - gy1)

    ix1 = torch.max(px1, gx1)
    iy1 = torch.max(py1, gy1)
    ix2 = torch.min(px2, gx2)
    iy2 = torch.min(py2, gy2)

    iw = (ix2 - ix1).clamp(min=0)
    ih = (iy2 - iy1).clamp(min=0)
    inter = iw * ih

    union = p_area + g_area - inter + eps
    iou = inter / union

    cx1 = torch.min(px1, gx1)
    cy1 = torch.min(py1, gy1)
    cx2 = torch.max(px2, gx2)
    cy2 = torch.max(py2, gy2)

    c_w = cx2 - cx1
    c_h = cy2 - cy1
    c2 = c_w ** 2 + c_h ** 2 + eps

    p_cx = (px1 + px2) / 2
    p_cy = (py1 + py2) / 2
    g_cx = (gx1 + gx2) / 2
    g_cy = (gy1 + gy2) / 2

    d2 = (p_cx - g_cx) ** 2 + (p_cy - g_cy) ** 2

    diou = 1 - iou + d2 / c2

    return diou.mean()

In [ ]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

In [ ]:
!pip install torchmetrics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 21.7 MB/s eta 0:00:00


In [ ]:
from torchmetrics.detection import MeanAveragePrecision

@torch.no_grad()
def validate(dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
    """ Метод для валидации модели.
    Возвращает mAP (0.5 ... 0.95).
    """
    self.model.eval()
    # Считаем метрику mAP с помощью функции из torchmetrics
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = self.model(images)
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)
        metric.update(predicts, targets)
    return metric.compute()["map"].item()


In [ ]:
import torchvision
import copy
import io
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchmetrics.detection import MeanAveragePrecision

class HaloDatasetXYXY(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)
        target = {"image_id": row["image_id"]}

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        labels = [label - 1 for label in labels]

        boxes_coco = row['bbox'].tolist()
        boxes_xyxy = [[x, y, x + w, y + h] for x, y, w, h in boxes_coco]

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes_xyxy, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)
            boxes = boxes_xyxy

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

train_transform = A.Compose([
    A.LongestMaxSize(max_size=640),
    A.PadIfNeeded(min_height=640, min_width=640, border_mode=0),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.3, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.3),
    A.GaussNoise(var_limit=(10, 50), p=0.3),
    A.MotionBlur(blur_limit=3, p=0.2),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.3))

test_transform = A.Compose([
    A.LongestMaxSize(max_size=640),
    A.PadIfNeeded(min_height=640, min_width=640, border_mode=0),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

train_dataset = HaloDatasetXYXY(df_train, transform=train_transform)
test_dataset = HaloDatasetXYXY(df_test, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

class ComputeLoss(nn.Module):
    def __init__(self, num_classes=4, alpha_focal=2.0, gamma_focal=2.0):
        super().__init__()
        self.num_classes = num_classes
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        self.alpha_focal = alpha_focal
        self.gamma_focal = gamma_focal

    def forward(self, bbox_preds, cls_logits, targets, model):
        device = bbox_preds.device
        bs = bbox_preds.size(0)

        decoded = model.decode_bboxes(bbox_preds)
        cls_probs = torch.sigmoid(cls_logits)

        total_box = torch.tensor(0.0, device=device)
        total_cls = torch.tensor(0.0, device=device)
        total_obj = torch.tensor(0.0, device=device)
        num_pos = 0

        for i in range(bs):
            gt = targets[i]
            gt_boxes = gt['boxes'].to(device)
            gt_labels = gt['labels'].to(device)

            assigned_gt_inds, assigned_labels, _ = TAL_assigner(
                decoded[i], cls_probs[i], gt_boxes, gt_labels,
                model.anchor_centers[0], alpha=6.0, beta=1.0, topk=13, device=device
            )

            pos_mask = assigned_gt_inds > 0
            num_pos += pos_mask.sum().item()
            if pos_mask.sum() == 0:
                continue

            pos_decoded = decoded[i][pos_mask]
            assigned_gt = assigned_gt_inds[pos_mask] - 1
            total_box += diou_loss(pos_decoded, gt_boxes[assigned_gt])

            pos_cls = cls_logits[i][pos_mask]
            target_cls = torch.zeros_like(pos_cls)
            for j, label in enumerate(assigned_labels[pos_mask]):
                target_cls[j, label] = 1.0

            p = torch.sigmoid(pos_cls)
            ce = self.bce(pos_cls, target_cls)
            fw = (target_cls * (1 - p) + (1 - target_cls) * p) ** self.gamma_focal
            total_cls += (self.alpha_focal * fw * ce).mean()

            neg_mask = ~pos_mask
            if neg_mask.sum() > 0:
                neg_cls = cls_logits[i][neg_mask]
                target_neg = torch.zeros_like(neg_cls)
                ce_neg = self.bce(neg_cls, target_neg)
                pt_neg = torch.sigmoid(neg_cls)
                total_obj += (pt_neg ** self.gamma_focal * ce_neg).mean() * 0.5

        if num_pos > 0:
            return (total_box + total_cls + total_obj) / bs
        return (total_cls + total_obj) / bs

def filter_predictions(outputs, score_threshold=0.01, nms_threshold=0.5, max_detections=300):
    bboxes, scores = outputs
    bs = bboxes.size(0)
    results = []
    for i in range(bs):
        max_scores, labels = scores[i].max(dim=1)
        keep = max_scores > score_threshold
        if keep.sum() == 0:
            results.append({"boxes": torch.zeros((0, 4), device=bboxes.device),
                          "scores": torch.zeros(0, device=bboxes.device),
                          "labels": torch.zeros(0, dtype=torch.int64, device=bboxes.device)})
            continue
        keep_idx = torchvision.ops.batched_nms(bboxes[i][keep], max_scores[keep], labels[keep], nms_threshold)
        keep_idx = keep_idx[:max_detections]
        results.append({"boxes": bboxes[i][keep][keep_idx],
                       "scores": max_scores[keep][keep_idx],
                       "labels": labels[keep][keep_idx]})
    return results

@torch.no_grad()
def validate(model, dataloader, device="cuda", score_threshold=0.01, nms_threshold=0.5):
    model.eval()
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = model(images)
        predicts = filter_predictions(outputs, score_threshold, nms_threshold)
        for t in targets:
            t['boxes'] = t['boxes'].to(device)
            t['labels'] = t['labels'].to(device)
        metric.update(predicts, targets)
    return metric.compute()["map"].item()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Detector(
    backbone_model_name="efficientnet_b0",
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=(32, 64, 128),
    anchor_ratios=(0.5, 1.0, 2.0),
    input_size=(640, 640),
).to(device)

backbone_params = [p for n, p in model.named_parameters() if 'backbone' in n and p.requires_grad]
other_params = [p for n, p in model.named_parameters() if 'backbone' not in n and p.requires_grad]

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 5e-5},
    {'params': other_params, 'lr': 1e-3},
], weight_decay=0.0005)

warmup_epochs = 5
total_epochs = 100

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    return 0.5 * (1 + torch.cos(torch.tensor((epoch - warmup_epochs) / (total_epochs - warmup_epochs) * 3.14159)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = ComputeLoss(num_classes=4)

best_map = 0
best_model = None

for epoch in range(total_epochs):
    model.train()
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{total_epochs}")
    for images, targets in pbar:
        images = images.to(device)
        optimizer.zero_grad()
        bbox_preds, cls_logits = model(images)
        loss = criterion(bbox_preds, cls_logits, targets, model)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        epoch_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)

    if (epoch + 1) % 5 == 0 or epoch == total_epochs - 1:
        current_map = validate(model, val_loader, device=device)
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, mAP={current_map:.4f}")
        if current_map > best_map:
            best_map = current_map
            best_model = copy.deepcopy(model.state_dict())
            print(f"  -> New best! mAP={best_map:.4f}")
    else:
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}")

if best_model is not None:
    model.load_state_dict(best_model)

final_map = validate(model, val_loader, device=device, score_threshold=0.01)
print(f"\nFinal mAP: {final_map:.4f}")
print(f"Best mAP: {best_map:.4f}")


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_17793/661627398.py:47: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50), p=0.3),
Epoch 1/100: 100%|██████████| 58/58 [01:10<00:00,  1.22s/it, loss=0.6567]


Epoch 1: Loss=0.7799


Epoch 2/100: 100%|██████████| 58/58 [01:12<00:00,  1.25s/it, loss=0.3476]


Epoch 2: Loss=0.6447


Epoch 3/100: 100%|██████████| 58/58 [01:14<00:00,  1.29s/it, loss=0.6637]


...

Epoch 87/100: 100%|██████████| 58/58 [01:10<00:00,  1.22s/it, loss=0.4082]


Epoch 87: Loss=0.3470


Epoch 88/100: 100%|██████████| 58/58 [01:10<00:00,  1.22s/it, loss=0.2050]


Epoch 88: Loss=0.3440


Epoch 89/100:  43%|████▎     | 25/58 [00:32<00:42,  1.30s/it, loss=0.3821]

Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

mAp>0.2